In [147]:
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.schema import Document
from langchain_core.output_parsers import StrOutputParser 

In [148]:
loader = TextLoader("/Users/pawanpahune/RAG_From_Scratch/Hybrid_Search_Techniques/langchain.txt")
doc = loader.load()

In [149]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
chunks = splitter.split_documents(doc)
chunks

[Document(metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Hybrid_Search_Techniques/langchain.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Hybrid_Search_Techniques/langchain.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Hybrid_Search_Techniques/langchain.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed 

In [150]:
query = "How can i use langchain to build LLM application using memory and tools"

In [151]:
from langchain_classic.vectorstores import FAISS
from langchain_classic.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks,embedding_model)
retriever = vector_store.as_retriever(search_kwargs = {'k':8})



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6209.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [152]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from langchain_classic.schema import Document
from langchain_classic.vectorstores import FAISS
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.schema.runnable import RunnableLambda, RunnableMap
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from dotenv import load_dotenv
load_dotenv()


os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [153]:
from langchain_classic.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_model('groq:openai/gpt-oss-120b', temperature=0.7)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x12e08ebd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12df571d0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [154]:
prompt = PromptTemplate.from_template(
    """You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
"{documents}"

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)"""
)

In [155]:
retrieved = retriever.invoke(query)

In [156]:
retrieved

[Document(id='31b5257b-8bbe-44e1-9bff-243169ecd5c9', metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Hybrid_Search_Techniques/langchain.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='9fef4cc0-c9a4-417e-8ae2-9bd17cb56e6d', metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Hybrid_Search_Techniques/langchain.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='7291c50d-a796-4643-83ef-768039503d05', metadata={'source': '/Users/

In [157]:
chain = prompt|llm|StrOutputParser()

In [158]:
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n"{documents}"\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)')
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x12e08ebd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12df571d0>, model_name='openai/gpt-oss-120

In [159]:
doc_lines = [f"{i+1}.{doc.page_content}"for i,doc in enumerate(retrieved) ]
doc_lines

['1.LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '2.LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '3.LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '4.Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM

In [160]:
formatted_docs = "\n".join(doc_lines)
doc_lines

['1.LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '2.LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '3.LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '4.Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM

In [161]:
formatted_docs

'1.LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\n2.LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.\n3.LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\n4.Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses

In [162]:
response = chain.invoke({"question": query, "documents": formatted_docs})

In [163]:
response

'2,1,5,3,4,6'

In [164]:
indices =[int(x.strip())-1 for x in response.split(",") if x.strip().isdigit()]
reranked_docs = [retrieved[i] for i in  indices if 0<= i < len(retrieved)]
indices

[1, 0, 4, 2, 3, 5]

In [165]:
reranked_docs

[Document(id='9fef4cc0-c9a4-417e-8ae2-9bd17cb56e6d', metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Hybrid_Search_Techniques/langchain.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='31b5257b-8bbe-44e1-9bff-243169ecd5c9', metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Hybrid_Search_Techniques/langchain.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='72ea83da-c0c9-45d1-a3ca-b72d46ec6eb5', metadata={'source': '/Users/

In [166]:
for i, doc in enumerate(reranked_docs):
    print(f"Rank {i+1}: {doc.page_content}\n")

Rank 1: LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.

Rank 2: LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.

Rank 3: FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed indexes, which makes it scalable for large document stores.
Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code executio